In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("parulpandey/palmer-archipelago-antarctica-penguin-data")

print("Path to dataset files:", path)

100%|██████████| 11.3k/11.3k [00:00<00:00, 11.8MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data/versions/1


In [2]:
#Import and Read the dataset
import pandas as pd
import numpy as np

URL="https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(URL)
print("Shape:",df.shape)
print(df.head())

Shape: (344, 7)
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   
1  Adelie  Torgersen            39.5           17.4              186.0   
2  Adelie  Torgersen            40.3           18.0              195.0   
3  Adelie  Torgersen             NaN            NaN                NaN   
4  Adelie  Torgersen            36.7           19.3              193.0   

   body_mass_g     sex  
0       3750.0    MALE  
1       3800.0  FEMALE  
2       3250.0  FEMALE  
3          NaN     NaN  
4       3450.0  FEMALE  


In [4]:
#Find missing values
print(df.isnull().sum())
print((df.isnull().sum()/len(df)*100).round(2))
print(df.dtypes)
df.info()
for c in ['species', 'island', 'sex']:
    print(c,"->",df[c].unique())

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64
species              0.00
island               0.00
bill_length_mm       0.58
bill_depth_mm        0.58
flipper_length_mm    0.58
body_mass_g          0.58
sex                  3.20
dtype: float64
species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_

In [6]:
#finding and handling missing values
df['species'] = df['species'].astype('category')
df['island']  = df['island'].astype('category')
df['sex']     = df['sex'].astype('category')


fully_missing = df[['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']].isnull().all(axis=1)
print("Fully-blank rows:", fully_missing.sum())
df = df[~fully_missing].reset_index(drop=True)


num_cols = ['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
for col in num_cols:
    df[col] = df.groupby('species')[col].transform(lambda s: s.fillna(s.median()))


sex_mode = df['sex'].mode()[0]
df['sex'] = df['sex'].fillna(sex_mode)


df['flipper_length_mm'] = df['flipper_length_mm'].astype(int)
df['body_mass_g']       = df['body_mass_g'].astype(int)

print(df.isnull().sum())
print(df.dtypes)

Fully-blank rows: 0
species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64
species              category
island               category
bill_length_mm        float64
bill_depth_mm         float64
flipper_length_mm       int64
body_mass_g             int64
sex                  category
dtype: object


/tmp/ipykernel_569/519480639.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[col] = df.groupby('species')[col].transform(lambda s: s.fillna(s.median()))


In [10]:
#normalization using scikitlearn

import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

print("Before normalization:")
print(df[num_cols].describe().round(2))

# Min-Max normalization -> scales every value into [0, 1]
minmax = MinMaxScaler()
df_minmax = df.copy()
df_minmax[num_cols] = minmax.fit_transform(df[num_cols])

# Standardization (Z-score) -> mean 0, std 1
standard = StandardScaler()
df_standard = df.copy()
df_standard[num_cols] = standard.fit_transform(df[num_cols])

print("\nAfter Min-Max normalization [0,1]:")
print(df_minmax[num_cols].describe().round(3))

print("\nAfter Standardization (mean~0, std~1):")
print(df_standard[num_cols].describe().round(3))

Before normalization:
       bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g
count          344.00         344.00             344.00       344.00
mean            43.92          17.15             200.93      4202.62
std              5.45           1.97              14.06       801.23
min             32.10          13.10             172.00      2700.00
25%             39.20          15.58             190.00      3550.00
50%             44.45          17.30             197.00      4050.00
75%             48.50          18.70             213.25      4756.25
max             59.60          21.50             231.00      6300.00

After Min-Max normalization [0,1]:
       bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g
count         344.000        344.000            344.000      344.000
mean            0.430          0.482              0.490        0.417
std             0.198          0.235              0.238        0.223
min             0.000          0.000         

In [11]:
# sex: binary, no order -> 0/1 mapping
df['sex'] = df['sex'].map({'MALE': 0, 'FEMALE': 1})

# island: nominal, 3 unordered categories -> one-hot encoding
df = pd.get_dummies(df, columns=['island'], prefix='island', drop_first=True)

# species: the natural class label for this dataset.
# Keep an integer-coded version (species_code, 0/1/2) for use as a classification
# target, AND one-hot encode it too, in case a model needs unordered dummy columns
# instead of an implied ranking.
df['species_code'] = df['species'].astype('category').cat.codes
df = pd.get_dummies(df, columns=['species'], prefix='species', drop_first=False)

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(df.dtypes)
print(df.head())

bill_length_mm        float64
bill_depth_mm         float64
flipper_length_mm       int64
body_mass_g             int64
sex                  category
island_Dream            int64
island_Torgersen        int64
species_code             int8
species_Adelie          int64
species_Chinstrap       int64
species_Gentoo          int64
dtype: object
   bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g sex  \
0            39.1           18.7                181         3750   0   
1            39.5           17.4                186         3800   1   
2            40.3           18.0                195         3250   1   
3            38.8           18.4                190         3700   0   
4            36.7           19.3                193         3450   1   

   island_Dream  island_Torgersen  species_code  species_Adelie  \
0             0                 1             0               1   
1             0                 1             0               1   
2             0       

In [12]:
"""
Data Preprocessing, Formatting, and Normalization
Dataset: Palmer Penguins (open-source)
Source : https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv

Run with:  python3 penguins_preprocessing_pipeline.py
Requires : pandas, numpy, scikit-learn
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# ---------------------------------------------------------------------------
# STEP 1: Import / read the dataset
# ---------------------------------------------------------------------------
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(URL)

print("=" * 70); print("STEP 1: LOAD DATA"); print("=" * 70)
print("Shape:", df.shape)
print(df.head())

# ---------------------------------------------------------------------------
# STEP 2: Missing values & raw dtypes
# ---------------------------------------------------------------------------
print("\n" + "=" * 70); print("STEP 2: MISSING VALUES & RAW DTYPES"); print("=" * 70)
print(df.isnull().sum())
print((df.isnull().sum() / len(df) * 100).round(2))
print(df.dtypes)
for c in ['species', 'island', 'sex']:
    print(c, "->", df[c].unique())

# ---------------------------------------------------------------------------
# STEP 3: Formatting - correct variable types
# ---------------------------------------------------------------------------
df['species'] = df['species'].astype('category')
df['island']  = df['island'].astype('category')
df['sex']     = df['sex'].astype('category')

# ---------------------------------------------------------------------------
# STEP 4: Handle missing values
# ---------------------------------------------------------------------------
# Rows where ALL measurements are blank carry no usable information -> drop
fully_missing = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']].isnull().all(axis=1)
df = df[~fully_missing].reset_index(drop=True)

# Impute remaining numeric NaNs with the MEDIAN WITHIN EACH SPECIES
# (species differ enough in size that a global median would be misleading)
num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
for col in num_cols:
    df[col] = df.groupby('species')[col].transform(lambda s: s.fillna(s.median()))

# sex: categorical -> impute with the mode
sex_mode = df['sex'].mode()[0]
df['sex'] = df['sex'].fillna(sex_mode)

# Now safe to cast whole-number measurement columns to int
df['flipper_length_mm'] = df['flipper_length_mm'].astype(int)
df['body_mass_g']       = df['body_mass_g'].astype(int)

print("\n" + "=" * 70); print("STEP 4: AFTER CLEANING"); print("=" * 70)
print(df.isnull().sum())
print(df.dtypes)
print("Shape:", df.shape)

# ---------------------------------------------------------------------------
# STEP 5: Categorical -> quantitative
# ---------------------------------------------------------------------------
df['sex'] = df['sex'].map({'MALE': 0, 'FEMALE': 1})

df = pd.get_dummies(df, columns=['island'], prefix='island', drop_first=True)

df['species_code'] = df['species'].astype('category').cat.codes  # integer target label
df = pd.get_dummies(df, columns=['species'], prefix='species', drop_first=False)

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print("\n" + "=" * 70); print("STEP 5: AFTER ENCODING"); print("=" * 70)
print(df.dtypes)
print(df.head())

# ---------------------------------------------------------------------------
# STEP 6: Normalization
# ---------------------------------------------------------------------------
num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

minmax = MinMaxScaler()
df_minmax = df.copy()
df_minmax[num_cols] = minmax.fit_transform(df[num_cols])

standard = StandardScaler()
df_standard = df.copy()
df_standard[num_cols] = standard.fit_transform(df[num_cols])

print("\n" + "=" * 70); print("STEP 6: NORMALIZATION"); print("=" * 70)
print("Before:\n", df[num_cols].describe().round(2))
print("\nMin-Max [0,1]:\n", df_minmax[num_cols].describe().round(3))
print("\nStandardized:\n", df_standard[num_cols].describe().round(3))

df.to_csv("penguins_encoded.csv", index=False)
df_minmax.to_csv("penguins_normalized_minmax.csv", index=False)
df_standard.to_csv("penguins_normalized_standard.csv", index=False)
print("\nSaved: penguins_encoded.csv, penguins_normalized_minmax.csv, penguins_normalized_standard.csv")

STEP 1: LOAD DATA
Shape: (344, 7)
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g     sex
0  Adelie  Torgersen            39.1           18.7              181.0       3750.0    MALE
1  Adelie  Torgersen            39.5           17.4              186.0       3800.0  FEMALE
2  Adelie  Torgersen            40.3           18.0              195.0       3250.0  FEMALE
3  Adelie  Torgersen             NaN            NaN                NaN          NaN     NaN
4  Adelie  Torgersen            36.7           19.3              193.0       3450.0  FEMALE

STEP 2: MISSING VALUES & RAW DTYPES
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64
species              0.00
island               0.00
bill_length_mm       0.58
bill_depth_mm        0.58
flipper_length_mm    0.58
body_mass_g          0.58
sex                  3.20
dtype: float64

/tmp/ipykernel_569/2082093352.py:55: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[col] = df.groupby('species')[col].transform(lambda s: s.fillna(s.median()))


In [14]:
# Save all the preprocessed versions
df.to_csv("penguins_cleaned.csv", index=False)        # after missing-value handling
df.to_csv("penguins_encoded.csv", index=False)        # after categorical -> numeric encoding
df_minmax.to_csv("penguins_normalized_minmax.csv", index=False)
df_standard.to_csv("penguins_normalized_standard.csv", index=False)

# Download each one to your computer
from google.colab import files
files.download("penguins_cleaned.csv")
files.download("penguins_encoded.csv")
files.download("penguins_normalized_minmax.csv")
files.download("penguins_normalized_standard.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
from google.colab import drive
drive.mount('/content/drive')

df.to_csv("/content/drive/MyDrive/penguins_cleaned.csv", index=False)
# etc.

Mounted at /content/drive


In [16]:
df.to_csv("penguins_cleaned.csv", index=False)